# 02 — Enriquecimiento y Unificación (Snowpark API)

Este notebook utiliza **Snowflake Snowpark** para transformar los datos. A diferencia de PySpark con el conector de Snowflake (que puede fallar y mover datos masivamente), Snowpark está diseñado específicamente para compilar el 100% del código a SQL nativo de Snowflake. Esto garantiza un **Query Pushdown perfecto** y cero Data Movement.

In [ ]:
import os
from snowflake.snowpark import Session
from snowflake.snowpark.functions import col, coalesce, when, lit

print("=" * 60)
print("CELDA 1: Configuración de Snowpark Session")
print("=" * 60)

connection_parameters = {
    "account":     os.environ['SF_ACCOUNT'],
    "user":        os.environ["SF_USER"],
    "password":    os.environ["SF_PASSWORD"],
    "database":    os.environ["SF_DATABASE"],
    "schema":      os.environ["SF_SCHEMA_RAW"],
    "warehouse":   os.environ["SF_WAREHOUSE"],
    "role":        os.environ["SF_ROLE"],
}

session = Session.builder.configs(connection_parameters).create()

print("✓ Snowpark Session Inicializada.")
print("✓ El procesamiento se ejecutará nativamente en Snowflake.")

## 2.1 Carga de Datos (Snowpark DataFrames)

In [2]:
print("=" * 60)
print("CELDA 2: Lectura de Tablas")
print("=" * 60)

df_raw = session.table("TRIPS_RAW")
df_zones = session.table("TAXI_ZONE_LOOKUP")

print("✓ DataFrames Lógicos Creados (Lazy evaluation)")

CELDA 2: Lectura de Tablas
✓ DataFrames Lógicos Creados (Lazy evaluation)


## 2.2 Transformación con Snowpark API
Aplicamos joins, coalesces y when/otherwise nativos de Snowpark.

In [ ]:
print("=" * 60)
print("CELDA 3: Transformaciones Snowpark")
print("=" * 60)

# Preparamos los DataFrames de Zonas para el Join
df_pu = df_zones.select(
    col("LocationID").alias("pu_loc_id"), 
    col("Zone").alias("pu_zone"), 
    col("Borough").alias("pu_borough")
)
df_do = df_zones.select(
    col("LocationID").alias("do_loc_id"), 
    col("Zone").alias("do_zone"), 
    col("Borough").alias("do_borough")
)

# Pipeline de Transformación
df_enriched = (
    df_raw
    # 1. Joins con Taxi Zones
    .join(df_pu, col("PULocationID") == col("pu_loc_id"), "left")
    .join(df_do, col("DOLocationID") == col("do_loc_id"), "left")
    
    # 2. Unificación de Fechas
    .with_column("pickup_datetime", coalesce(col("tpep_pickup_datetime"), col("lpep_pickup_datetime")))
    .with_column("dropoff_datetime", coalesce(col("tpep_dropoff_datetime"), col("lpep_dropoff_datetime")))
    
    # 3. Decodificación de Catálogos
    .with_column("vendor_name", 
                when(col("VendorID") == 1, lit("Creative Mobile Technologies"))
                .when(col("VendorID") == 2, lit("VeriFone Inc."))
                .otherwise(lit("Unknown")))
    
    .with_column("rate_code_id", col("RatecodeID").cast("integer"))
    .with_column("rate_code_desc", 
                when(col("rate_code_id") == 1, lit("Standard rate"))
                .when(col("rate_code_id") == 2, lit("JFK"))
                .when(col("rate_code_id") == 3, lit("Newark"))
                .when(col("rate_code_id") == 4, lit("Nassau/Westchester"))
                .when(col("rate_code_id") == 5, lit("Negotiated fare"))
                .when(col("rate_code_id") == 6, lit("Group ride"))
                .otherwise(lit("Unknown")))
    
    .with_column("payment_type_desc", 
                when(col("payment_type") == 1, lit("Credit card"))
                .when(col("payment_type") == 2, lit("Cash"))
                .when(col("payment_type") == 3, lit("No charge"))
                .when(col("payment_type") == 4, lit("Dispute"))
                .when(col("payment_type") == 5, lit("Unknown"))
                .when(col("payment_type") == 6, lit("Voided trip"))
                .otherwise(lit("Other")))
    
    # 4. Selección Final de Columnas
    .select(
        "pickup_datetime", "dropoff_datetime",
        col("PULocationID").alias("pu_location_id"), "pu_zone", "pu_borough",
        col("DOLocationID").alias("do_location_id"), "do_zone", "do_borough",
        "service_type",
        col("VendorID").alias("vendor_id"), "vendor_name",
        "rate_code_id", "rate_code_desc",
        "payment_type", "payment_type_desc",
        "trip_type", "passenger_count", "trip_distance", "store_and_fwd_flag",
        "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
        "improvement_surcharge", "congestion_surcharge",
        col("Airport_fee").alias("airport_fee"), "total_amount",
        "run_id", "ingested_at_utc", "source_year", "source_month"
    )
)

print("✓ Grafo de transformaciones construido en memoria.")

## 2.3 Ejecución y Escritura (Pushdown Real)
Como usamos Snowpark, al ejecutar `save_as_table`, el framework envía toda la lógica compilada como un único statement SQL a Snowflake.

In [ ]:
print("=" * 60)
print("CELDA 4: Ejecución de Escritura")
print("=" * 60)

print("\n>>> Escribiendo TRIPS_ENRICHED en Snowflake...")

# Muestra la consulta SQL generada por Snowpark antes de ejecutarla (Opcional, para debug)
print("Query generada:")
print(df_enriched.queries.get("queries")[0][:500] + "...")

df_enriched.write.mode("overwrite").save_as_table("TRIPS_ENRICHED")

print("✓ Tabla TRIPS_ENRICHED materializada con éxito en Snowflake.")
print("=" * 60)